# 10.5 Running scripts

In [1]:
# install dependencies
%pip install -q amplpy pandas

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

A sequence of commands that you want to use again and again should be written down once, and then run with a single command. In Python, the natural way to capture such a sequence is a *function*. A function can be defined in a notebook cell, or stored together with others in a *module*, a file of Python code that any script or notebook can import. In this section we use functions and modules as the Python counterparts of AMPL's command files.

```{admonition} Curiosity: AMPL's include and commands
:class: tip

AMPL provides several commands that cause input to be taken from a file. The command `include filename` is replaced by the contents of the named file, and the statement `commands filename;` is very similar. The `model` and `data` commands are special cases of `include` that put the command interpreter into model or data mode before reading the specified file. A file of commands, such as a `dietu.run` file containing `model dietu.mod; data dietu.dat; solve; display Buy;`, could then be run in one operation by `include dietu.run;`.

In amplpy, the `read()` method interprets any such file, whether it contains model declarations, commands, or both.
```

As an illustration, the following functions load the diet model `dietu.mod` of [Section 10.3](10_3_modifying_data.ipynb) and its data, run the problem, and display the optimal values of the variables:

In [2]:
%%writefile dietu.mod

set MINREQ;                     # nutrients with minimum requirements
set MAXREQ;                     # nutrients with maximum requirements
set NUTR = MINREQ union MAXREQ; # nutrients
set FOOD;                       # foods

param cost {FOOD} > 0;
param f_min {FOOD} >= 0;
param f_max {j in FOOD} >= f_min[j];

param n_min {MINREQ} >= 0;
param n_max {MAXREQ} >= 0;

param amt {NUTR,FOOD} >= 0;

var Buy {j in FOOD} >= f_min[j], <= f_max[j];

minimize Total_Cost: sum {j in FOOD} cost[j] * Buy[j];

subject to Diet_Min {i in MINREQ}:
   sum {j in FOOD} amt[i,j] * Buy[j] >= n_min[i];

subject to Diet_Max {i in MAXREQ}:
   sum {j in FOOD} amt[i,j] * Buy[j] <= n_max[i];

Writing dietu.mod


In [3]:
import pandas as pd

food_df = pd.DataFrame(
    [
        ("BEEF", 3.19, 2, 10),
        ("CHK",  2.59, 2, 10),
        ("FISH", 2.29, 2, 10),
        ("HAM",  2.89, 2, 10),
        ("MCH",  1.89, 2, 10),
        ("MTL",  1.99, 2, 10),
        ("SPG",  1.99, 2, 10),
        ("TUR",  2.49, 2, 10),
    ],
    columns=["FOOD", "cost", "f_min", "f_max"],
).set_index("FOOD")

MINREQ = ["A", "B1", "B2", "C", "CAL"]
MAXREQ = ["A", "NA", "CAL"]

n_min = {"A": 700, "B1": 0, "B2": 0, "C": 700, "CAL": 16000}
n_max = {"A": 20000, "NA": 50000, "CAL": 24000}

# Nutrient content: one row per nutrient, one column per food
amt_df = pd.DataFrame(
    {
        "BEEF": [60, 20, 10, 15,  938, 295],
        "CHK":  [ 8,  0, 20, 20, 2180, 770],
        "FISH": [ 8, 10, 15, 10,  945, 440],
        "HAM":  [40, 40, 35, 10,  278, 430],
        "MCH":  [15, 35, 15, 15, 1182, 315],
        "MTL":  [70, 30, 15, 15,  896, 400],
        "SPG":  [25, 50, 25, 15, 1329, 370],
        "TUR":  [60, 20, 15, 10, 1397, 450],
    },
    index=["A", "C", "B1", "B2", "NA", "CAL"],
)
amt = amt_df.stack().to_dict()   # {(nutrient, food): amount}

In [4]:
def load_dietu(ampl):
    """Read the dietu model and assign its data."""
    ampl.read("dietu.mod")
    ampl.set_data(food_df, "FOOD")
    ampl.set["MINREQ"] = MINREQ
    ampl.set["MAXREQ"] = MAXREQ
    ampl.param["n_min"] = n_min
    ampl.param["n_max"] = n_max
    ampl.param["amt"] = amt


def run_dietu(ampl):
    """Load dietu, solve it, and display the purchases."""
    load_dietu(ampl)
    ampl.solve(solver="highs")
    assert ampl.solve_result == "solved", ampl.solve_result

    ampl.option["display_1col"] = 5
    ampl.option["display_round"] = 1
    ampl.display("Buy")

Calling `run_dietu()` will then load the model and data, run the problem, and display the optimal values of the variables:

In [5]:
run_dietu(ampl)

HiGHS 1.15.1: optimal solution; objective 74.27382022
2 simplex iterations
0 barrier iterations
Buy [*] :=
BEEF  2.0   FISH  2.0    MCH  2.0    SPG  5.3
 CHK 10.0    HAM  2.0    MTL  6.2    TUR  2.0
;



When a function itself calls another function, execution of the first is suspended while the second one runs. In this example, the call to `run_dietu()` causes the subsequent call of `load_dietu()`, which can also be used by itself whenever the model and data are needed without solving.

One particularly useful kind of function contains a list of option settings that you want to apply before any other commands, to modify the default options. Stored in a module, such a function can be imported and called at the beginning of every script, so that your option settings are the same each time.

To illustrate further, consider how we might perform a simple sensitivity analysis on the multi-period production problem of [Section 4.2](../04/4_2_a_multiperiod_production_model.ipynb). Only 32 hours of production time are available in week 3, compared to 40 hours in the other weeks. Suppose that we want to see how much extra profit could be gained for each extra hour in week 3. We can accomplish this by repeatedly solving, recording the solution values, and increasing `avail[3]`:

In [6]:
%%writefile steelT.mod

set PROD;     # products
param T > 0;  # number of weeks

param rate {PROD} > 0;          # tons per hour produced
param inv0 {PROD} >= 0;         # initial inventory
param avail {1..T} >= 0;        # hours available in week
param market {PROD,1..T} >= 0;  # limit on tons sold in week

param prodcost {PROD} >= 0;     # cost per ton produced
param invcost {PROD} >= 0;      # carrying cost/ton of inventory
param revenue {PROD,1..T} >= 0; # revenue per ton sold

var Make {PROD,1..T} >= 0;      # tons produced
var Inv {PROD,0..T} >= 0;       # tons inventoried
var Sell {p in PROD, t in 1..T} >= 0, <= market[p,t]; # tons sold

# Total revenue less costs in all weeks
maximize Total_Profit:
   sum {p in PROD, t in 1..T} (revenue[p,t]*Sell[p,t] -
      prodcost[p]*Make[p,t] - invcost[p]*Inv[p,t]);

# Total of hours used by all products
# may not exceed hours available, in each week
subject to Time {t in 1..T}:
   sum {p in PROD} (1/rate[p]) * Make[p,t] <= avail[t];

# Initial inventory must equal given value
subject to Init_Inv {p in PROD}:  Inv[p,0] = inv0[p];

# Tons produced and taken from inventory
# must equal tons sold and put into inventory
subject to Balance {p in PROD, t in 1..T}:
   Make[p,t] + Inv[p,t-1] = Sell[p,t] + Inv[p,t];

Writing steelT.mod


In [7]:
import pandas as pd

steelT_prod = pd.DataFrame(
    [
        ["bands", 200, 10, 10, 2.5],
        ["coils", 140,  0, 11, 3],
    ],
    columns=["PROD", "rate", "inv0", "prodcost", "invcost"],
).set_index("PROD")

steelT_week = pd.DataFrame(
    [
        ["bands", 1, 25, 6000],
        ["bands", 2, 26, 6000],
        ["bands", 3, 27, 4000],
        ["bands", 4, 27, 6500],
        ["coils", 1, 30, 4000],
        ["coils", 2, 35, 2500],
        ["coils", 3, 37, 3500],
        ["coils", 4, 39, 4200],
    ],
    columns=["PROD", "T", "revenue", "market"],
).set_index(["PROD", "T"])

steelT_avail = {1: 40, 2: 40, 3: 32, 4: 40}

In [8]:
def load_steelT(ampl):
    """Read the steelT model and assign its data."""
    ampl.read("steelT.mod")
    ampl.param["T"] = 4
    ampl.param["avail"] = steelT_avail
    ampl.set_data(steelT_prod, "PROD")
    ampl.set_data(steelT_week)

In [9]:
ampl.reset()
load_steelT(ampl)

ampl.solve(solver="highs")
assert ampl.solve_result == "solved", ampl.solve_result

with open("steelT.sens", "w") as sens:
    print("Total_Profit =", ampl.obj["Total_Profit"].value(), file=sens)
    for name in ["Make", "Sell", "Inv"]:
        print(ampl.var[name].to_pandas().unstack().to_string(), file=sens)
    print(file=sens)

ampl.param["avail"][3] = ampl.param["avail"][3] + 5

HiGHS 1.15.1: optimal solution; objective 515033
16 simplex iterations
0 barrier iterations


To continue trying values of `avail[3]` in steps of 5 up to say 47, we must complete another three solve cycles in the same way. We can avoid writing the same commands over and over by creating a module containing the commands to be repeated, as a function:

In [10]:
%%writefile steelT_sens.py

def sens_step(ampl, filename="steelT.sens"):
    """Solve steelT, append the results to a file, and add 5 hours to week 3."""
    ampl.solve(solver="highs")
    assert ampl.solve_result == "solved", ampl.solve_result

    with open(filename, "a") as sens:
        print("Total_Profit =", ampl.obj["Total_Profit"].value(), file=sens)
        for name in ["Make", "Sell", "Inv"]:
            print(ampl.var[name].to_pandas().unstack().to_string(), file=sens)
        print(file=sens)

    ampl.param["avail"][3] = ampl.param["avail"][3] + 5

Writing steelT_sens.py


If we call this file `steelT_sens.py`, we can import the function from it, and then execute all of its commands with a single call to `sens_step()`:

In [11]:
from steelT_sens import sens_step

sens_step(ampl)
sens_step(ampl)
sens_step(ampl)

HiGHS 1.15.1: optimal solution; objective 532033
0 simplex iterations
0 barrier iterations
HiGHS 1.15.1: optimal solution; objective 549033
0 simplex iterations
0 barrier iterations
HiGHS 1.15.1: optimal solution; objective 565193
1 simplex iterations
0 barrier iterations


All of the solution values are written to the file `steelT.sens`, although we could just as well have printed them. Here are the first lines of the file, recorded for the original 32 hours in week 3:

In [12]:
with open("steelT.sens") as sens:
    print("".join(sens.readlines()[:13]))

Total_Profit = 515033.0
       Make.val                  
index1        1     2     3     4
index0                           
bands      5990  6000  1400  2000
coils      1407  1400  3500  4200
       Sell.val                  
index1        1     2     3     4
index0                           
bands      6000  6000  1400  2000
coils       307  2500  3500  4200
       Inv.val               
index1       0     1  2  3  4



Because `sens_step()` is stored in a module, any other notebook or script can import and use it. (If you edit a module after importing it, restart the Python kernel, or use `importlib.reload()`, so that the new version is used.) The next section shows how the repeated calls themselves can be replaced by a loop.